# Climate and Agriculture in Pakistan
### An exploratory data analysis of temperature, climate extremes and crop yields

**Sumbal Fatima** · BSc Agricultural Engineering, University of Agriculture, Faisalabad

![hero](../figures/00_hero.png)

This notebook asks three questions using public Kaggle data:

1. **Climate extremes:** how do record temperatures and rainfall differ between Peshawar and Gilgit?
2. **Long-term change:** has Pakistan's temperature changed, and when did the shift happen?
3. **Crops:** how have Pakistan's yields changed since 1990, and how do they compare with other countries?

Every number quoted in the text below is computed in the notebook, so results can be re-checked by re-running it. It is exploratory analysis: trend tests show *whether* something changed, not *why*.

**Contents:** 0 Setup · 1 Data overview and cleaning · 2 Climate extremes · 3 Pakistan temperature trend · 4 Crop yields and Pakistan's crops · 5 Conclusions and limitations

## 0. Setup

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns, pymannkendall as mk
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.gridspec import GridSpec

# One consistent palette across the notebook
INK, MUTED, SURF, GRID = '#0b0b0b', '#52514e', '#fcfcfb', '#e7e6e2'
BLUE, ORANGE, AQUA, RED, GREY = '#2a78d6', '#eb6834', '#1baf7a', '#e34948', '#b9b8b3'
DIV = LinearSegmentedColormap.from_list('div', [BLUE, '#f0efec', RED])      # cool-neutral-warm, for anomalies
mpl.rcParams.update({'figure.facecolor':SURF,'axes.facecolor':SURF,'savefig.facecolor':SURF,'axes.edgecolor':GRID,'axes.grid':True,
    'grid.color':GRID,'grid.linewidth':.8,'axes.spines.top':False,'axes.spines.right':False,'text.color':INK,'axes.labelcolor':MUTED,
    'xtick.color':MUTED,'ytick.color':MUTED,'axes.titleweight':'bold','axes.titlesize':12.5,'axes.titlelocation':'left','font.size':10.5,
    'figure.dpi':75,'savefig.bbox':'tight','legend.frameon':False})
FIG = '../figures/'
def save(name): plt.savefig(FIG+name, dpi=160); plt.show()

## 1. Data overview and cleaning
Before any analysis we look at each raw file and clean it. Every step is recorded in a **cleaning log** with row counts before and after, so each change can be checked. The raw CSV files are never modified.

### 1.1 Raw data: first look

In [2]:
raw = {'Peshawar record table': pd.read_csv('../data/peshawar.csv'), 'Gilgit record table': pd.read_csv('../data/gilgit.csv'),
       'Temperature by station (temp.csv)': pd.read_csv('../data/temp.csv'), 'Crop yield (yield_df.csv)': pd.read_csv('../data/yield_df.csv', index_col=0)}
ov = pd.DataFrame({name: {'rows': len(df), 'columns': df.shape[1], 'missing cells': int(df.isna().sum().sum()), 'exact duplicate rows': int(df.duplicated().sum())} for name, df in raw.items()}).T
ov

,rows,columns,missing cells,exact duplicate rows
Peshawar record table,13,7,0,0
Gilgit record table,13,7,0,0
Temperature by station (temp.csv),71311,3,2547,6958
Crop yield (yield_df.csv),28242,7,0,2310


In [3]:
pd.set_option('display.max_colwidth', 80); pd.set_option('display.width', 200)
for name, df in raw.items():
    print('=' * 70); print(name, '| column types:', dict(df.dtypes.astype(str))); display(df.head(3))

Peshawar record table | column types: {'Month': 'str', 'Highest Maximum Temperature in Celsius ': 'float64', 'Maximum Temperature date': 'str', 'Lowest Minimum Temperature in Celsius ': 'float64', 'Minimum Temperature date': 'str', 'Monthly Heaviest Rainfall in mm': 'float64', 'Heaviest Rainfall date': 'int64'}


,Month,Highest Maximum Temperature in Celsius,Maximum Temperature date,Lowest Minimum Temperature in Celsius,Minimum Temperature date,Monthly Heaviest Rainfall in mm,Heaviest Rainfall date
0,January,27.0,30-2007,-3.9,07-1970,150.0,1999
1,February,30.5,24-2016,-2.0,19-1905,236.0,2007
2,March,37.0,26-1892,-1.0,01-1905,222.6,1978


Gilgit record table | column types: {'Month': 'str', 'Highest Maximum Temperature in Celsius ': 'float64', 'Maximum Temperature date': 'str', 'Lowest Minimum Temperature in Celsius ': 'float64', 'Minimum Temperature date': 'str', 'Monthly Heaviest Rainfall in mm': 'float64', 'Heaviest Rainfall date': 'int64'}


,Month,Highest Maximum Temperature in Celsius,Maximum Temperature date,Lowest Minimum Temperature in Celsius,Minimum Temperature date,Monthly Heaviest Rainfall in mm,Heaviest Rainfall date
0,January,17.5,31-2007,-10.0,07-1972,32.2,2009
1,February,22.0,22-1993,-8.9,01-1964,35.5,2011
2,March,31.0,18-2022,-3.0,06-2000,73.7,1973


Temperature by station (temp.csv) | column types: {'year': 'int64', 'country': 'str', 'avg_temp': 'float64'}


,year,country,avg_temp
0,1849,Côte D'Ivoire,25.58
1,1850,Côte D'Ivoire,25.52
2,1851,Côte D'Ivoire,25.67


Crop yield (yield_df.csv) | column types: {'Area': 'str', 'Item': 'str', 'Year': 'int64', 'hg/ha_yield': 'int64', 'average_rain_fall_mm_per_year': 'float64', 'pesticides_tonnes': 'float64', 'avg_temp': 'float64'}


,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37


**Issues spotted in the raw data**
- **Climate tables:** some column names have trailing spaces, dates are text such as `30-2007` (day-year), and each file mixes a yearly summary row (`Annual`) in with the 12 monthly rows.
- **`temp.csv`:** some temperatures are missing, each country-year has several station rows, many of them exact repeats (a typical Pakistan year has 9 rows but only 6 distinct values), and the early Pakistan values are implausible (checked in section 3).
- **`yield_df.csv`:** many repeated rows, a column name with a slash (`hg/ha_yield`), and a rainfall column that never changes within a country.

### 1.2 Cleaning steps and cleaning log

In [4]:
log = []
def step(dataset, action, before, after, why): log.append(dict(dataset=dataset, step=action, rows_before=int(before), rows_after=int(after), why=why))

# --- climate record tables (Peshawar, Gilgit) ---
def clean_climate(df, city):
    d = df.copy(); d.columns = [c.strip() for c in d.columns]
    d = d.rename(columns={'Highest Maximum Temperature in Celsius':'record_max_c','Lowest Minimum Temperature in Celsius':'record_min_c',
        'Monthly Heaviest Rainfall in mm':'record_rain_mm','Maximum Temperature date':'max_date','Minimum Temperature date':'min_date','Heaviest Rainfall date':'rain_year'})
    d['max_year'] = d.max_date.str.split('-').str[-1].astype(int); d['min_year'] = d.min_date.str.split('-').str[-1].astype(int)
    d['rain_year'] = pd.to_numeric(d.rain_year, errors='coerce'); d['city'] = city
    d['month'] = pd.Categorical(d.Month, ['January','February','March','April','May','June','July','August','September','October','November','December','Annual'], ordered=True)
    return d
n_raw = len(raw['Peshawar record table']) + len(raw['Gilgit record table'])
cl_all = pd.concat([clean_climate(raw['Peshawar record table'], 'Peshawar'), clean_climate(raw['Gilgit record table'], 'Gilgit')], ignore_index=True)
step('Peshawar + Gilgit', 'Strip trailing spaces from column names and rename them (e.g. record_max_c)', n_raw, len(cl_all), 'Original names were long and some had trailing spaces')
step('Peshawar + Gilgit', 'Extract the year from the date text (30-2007 becomes 2007) and add a city column', n_raw, len(cl_all), 'Needed to ask when each record was set')
annual = cl_all[cl_all.Month=='Annual'].copy(); cl = cl_all[cl_all.Month!='Annual'].reset_index(drop=True); cl['month'] = cl.month.cat.remove_unused_categories()
step('Peshawar + Gilgit', 'Separate the Annual summary row from the 12 monthly rows', len(cl_all), len(cl), 'The Annual row is an all-year record, not a month, so it is analysed separately')

# --- temperature by station (temp.csv) ---
t = raw['Temperature by station (temp.csv)']
pk_all = t[t.country=='Pakistan']; step('temp.csv', 'Keep Pakistan stations only', len(t), len(pk_all), 'This project is about Pakistan')
pk = pk_all.dropna(subset=['avg_temp']); step('temp.csv', 'Drop rows with missing temperature', len(pk_all), len(pk), f'{len(pk_all)-len(pk)} Pakistan rows had no temperature value')
pk_u = pk.drop_duplicates(); step('temp.csv', 'Drop exact repeated rows (same year and same temperature)', len(pk), len(pk_u), f'{len(pk)-len(pk_u)} exact repeats; most years have up to 9 rows but only up to 6 distinct values, and repeats would count some readings twice')
pk_y = pk_u.groupby('year').agg(temp=('avg_temp','mean'), n_obs=('avg_temp','size')).reset_index()
step('temp.csv', 'Average the readings into one national value per year', len(pk_u), len(pk_y), 'Several readings per year; trend tests need one value per year')
step('temp.csv', 'Mark years before 1870 as unreliable (kept in the full-record chart, excluded from every trend test)', len(pk_y), (pk_y.year>=1870).sum(), 'Values swing between about 17.5 and 27.5 degrees C, which is not plausible for a national mean')

# --- crop yield (yield_df.csv) ---
y = raw['Crop yield (yield_df.csv)'].rename(columns={'hg/ha_yield':'yield_hgha','average_rain_fall_mm_per_year':'rain_mm','pesticides_tonnes':'pesticides_t'})
step('yield_df.csv', 'Rename columns (hg/ha_yield to yield_hgha, and so on)', len(y), len(y), 'Names without slashes are easier to use in code')
yc = (y.groupby(['Area','Item','Year'], as_index=False).agg(yield_hgha=('yield_hgha','first'), pesticides_t=('pesticides_t','first'), avg_temp=('avg_temp','mean'), rain_mm=('rain_mm','first')))
step('yield_df.csv', 'Remove repeated rows: keep one row per country, crop and year (temperature averaged)', len(y), len(yc), 'Station temperatures were joined and copied across each crop-year, repeating rows')
yc['yield_t_ha'] = yc.yield_hgha/10000
step('yield_df.csv', 'Convert yield from hg/ha to t/ha (divide by 10,000)', len(yc), len(yc), '1 tonne = 10,000 hectograms; t/ha is easier to read')
const = int((y.groupby('Area').rain_mm.nunique()==1).sum())
step('yield_df.csv', 'Do not use the rainfall column for analysis', len(yc), len(yc), f'It holds one fixed value per country: {const} of {y.Area.nunique()} countries show no year-to-year change')
pd.DataFrame(log)

,dataset,step,rows_before,rows_after,why
0,Peshawar + Gilgit,Strip trailing spaces from column names and rename them (e.g. record_max_c),26,26,Original names were long and some had trailing spaces
1,Peshawar + Gilgit,Extract the year from the date text (30-2007 becomes 2007) and add a city co...,26,26,Needed to ask when each record was set
2,Peshawar + Gilgit,Separate the Annual summary row from the 12 monthly rows,26,24,"The Annual row is an all-year record, not a month, so it is analysed separately"
3,temp.csv,Keep Pakistan stations only,71311,1805,This project is about Pakistan
4,temp.csv,Drop rows with missing temperature,1805,1637,168 Pakistan rows had no temperature value
5,temp.csv,Drop exact repeated rows (same year and same temperature),1637,1105,532 exact repeats; most years have up to 9 rows but only up to 6 distinct va...
6,temp.csv,Average the readings into one national value per year,1105,212,Several readings per year; trend tests need one value per year
7,temp.csv,"Mark years before 1870 as unreliable (kept in the full-record chart, exclude...",212,144,"Values swing between about 17.5 and 27.5 degrees C, which is not plausible f..."
8,yield_df.csv,"Rename columns (hg/ha_yield to yield_hgha, and so on)",28242,28242,Names without slashes are easier to use in code
9,yield_df.csv,"Remove repeated rows: keep one row per country, crop and year (temperature a...",28242,13130,"Station temperatures were joined and copied across each crop-year, repeating..."


### 1.3 Outlier and missing-value checks

In [5]:
def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1; return int(((s < q1 - 1.5*i) | (s > q3 + 1.5*i)).sum())
d_ok = pk_y[pk_y.year>=1870]
chk = pd.DataFrame([
    dict(series='Crop yield, log10, within each crop', values=len(yc), outliers_by_1_5_IQR=int(yc.groupby('Item').yield_hgha.apply(lambda s: iqr_outliers(np.log10(s))).sum()), decision='Kept: genuine differences between countries and years, not errors'),
    dict(series='Pakistan annual temperature, 1870-2013', values=len(d_ok), outliers_by_1_5_IQR=iqr_outliers(d_ok.temp), decision='Kept: every value is physically plausible'),
    dict(series='Pakistan annual temperature, before 1870', values=int((pk_y.year<1870).sum()), outliers_by_1_5_IQR=None, decision='Excluded from trend tests: implausible swings (see section 3)')])
display(chk)
from scipy.stats import theilslopes
sl = {lab: theilslopes(s_[s_.index>=1900].values, s_[s_.index>=1900].index.values)[0]*10 for lab, s_ in [('with repeated rows', pk.groupby('year').avg_temp.mean()), ('repeated rows removed', pk_u.groupby('year').avg_temp.mean())]}
print('Sensitivity check, Pakistan warming trend since 1900 (degrees C per decade):', {k: round(v, 3) for k, v in sl.items()})
display(pd.Series({'climate tables': int(cl.isna().sum().sum()), 'Pakistan temperature': int(pk_y.isna().sum().sum()), 'crop yields': int(yc.isna().sum().sum())}, name='missing cells after cleaning').to_frame())

,series,values,outliers_by_1_5_IQR,decision
0,"Crop yield, log10, within each crop",13130,215.0,"Kept: genuine differences between countries and years, not errors"
1,"Pakistan annual temperature, 1870-2013",144,1.0,Kept: every value is physically plausible
2,"Pakistan annual temperature, before 1870",68,NaN,Excluded from trend tests: implausible swings (see section 3)


Sensitivity check, Pakistan warming trend since 1900 (degrees C per decade): {'with repeated rows': np.float64(0.109), 'repeated rows removed': np.float64(0.108)}


,missing cells after cleaning
climate tables,0
Pakistan temperature,0
crop yields,0


### 1.4 Clean crop data: summary

In [6]:
print('Clean crop data:', yc.shape, '| countries:', yc.Area.nunique(), '| crops:', yc.Item.nunique(), '| years', yc.Year.min(), '-', yc.Year.max())
yc[['yield_t_ha','pesticides_t','avg_temp','rain_mm']].describe().round(2)

Clean crop data: (13130, 8) | countries: 101 | crops: 10 | years 1990 - 2013


,yield_t_ha,pesticides_t,avg_temp,rain_mm
count,13130.00,13130.00,13130.00,13130.00
mean,7.10,14838.70,19.84,1157.24
std,7.91,33652.45,6.66,743.62
min,0.00,0.04,1.30,51.00
25%,1.80,264.53,15.67,608.00
50%,3.95,2172.20,20.62,1083.00
75%,9.72,13335.22,25.78,1651.00
max,50.14,367778.00,30.42,3240.00


## 2. Climate extremes: Peshawar vs Gilgit
Each file holds one row per month with the **record** highest maximum, lowest minimum and heaviest rainfall, plus when each was set. They are record tables, not yearly series, so this section is descriptive.

In [7]:
print('Monthly rows:', len(cl), '| missing values:', int(cl.isna().sum().sum()), '| the Annual row of each file (overall records) is shown here')
annual[['city','record_max_c','max_year','record_min_c','min_year','record_rain_mm','rain_year']].round(1).reset_index(drop=True)

Monthly rows: 24 | missing values: 0 | the Annual row of each file (overall records) is shown here


,city,record_max_c,max_year,record_min_c,min_year,record_rain_mm,rain_year
0,Peshawar,50.0,1995,-3.9,1970,904.5,2003
1,Gilgit,46.3,1997,-11.1,1984,267.5,2010


In [8]:
months = list(cl.month.cat.categories); x = np.arange(12)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for ax, (city, col) in zip(axes, [('Peshawar', BLUE), ('Gilgit', ORANGE)]):
    d = cl[cl.city==city].sort_values('month')
    ax.fill_between(x, d.record_min_c, d.record_max_c, color=col, alpha=.16, lw=0)
    ax.plot(x, d.record_max_c, color=col, lw=2.2, marker='o', ms=5); ax.plot(x, d.record_min_c, color=col, lw=2.2, marker='o', ms=5, ls='--')
    ax.axhline(0, color=MUTED, lw=.8)
    ax.text(11.25, d.record_max_c.iloc[-1], 'record high', va='center', color=MUTED, fontsize=9.5); ax.text(11.25, d.record_min_c.iloc[-1], 'record low', va='center', color=MUTED, fontsize=9.5)
    ax.set_xticks(x); ax.set_xticklabels([m[:3] for m in months]); ax.set_xlim(-.4, 13.2); ax.set_title(city)
axes[0].set_ylabel('Record temperature (°C)'); fig.suptitle('The temperature range each city has ever recorded, by month', x=.01, ha='left', fontweight='bold', fontsize=13.5, y=1.02)
plt.tight_layout(); save('01_record_temperature_range.png')
hi = annual.set_index('city'); print(f"Hottest ever: Peshawar {hi.loc['Peshawar','record_max_c']} °C ({hi.loc['Peshawar','max_year']}), Gilgit {hi.loc['Gilgit','record_max_c']} °C ({hi.loc['Gilgit','max_year']})")
print(f"Coldest ever: Peshawar {hi.loc['Peshawar','record_min_c']} °C, Gilgit {hi.loc['Gilgit','record_min_c']} °C")
r = cl.groupby('city').apply(lambda g: g.record_max_c.max()-g.record_min_c.min()); print('Full record span (hottest minus coldest):', r.round(1).to_dict())

Hottest ever: Peshawar 50.0 °C (1995), Gilgit 46.3 °C (1997)
Coldest ever: Peshawar -3.9 °C, Gilgit -11.1 °C
Full record span (hottest minus coldest): {'Gilgit': 57.4, 'Peshawar': 53.9}


![01_record_temperature_range.png](../figures/01_record_temperature_range.png)

**Reading it:** Gilgit's band sits lower and is much wider in winter, with record lows near -11 °C, while Peshawar stays above freezing in most months. Both rise to a summer peak.

In [9]:
fig, ax = plt.subplots(figsize=(10, 4.4)); w = .38
for k, (city, col) in enumerate([('Peshawar', BLUE), ('Gilgit', ORANGE)]):
    d = cl[cl.city==city].sort_values('month'); ax.bar(x + (k-.5)*w, d.record_rain_mm, w-.04, color=col, label=city)
ax.set_xticks(x); ax.set_xticklabels([m[:3] for m in months]); ax.set_ylabel('Heaviest monthly rainfall on record (mm)'); ax.legend(loc='upper left')
ax.set_title('Rainfall extremes: Peshawar\'s monsoon months stand out'); ax.grid(axis='x', visible=False); save('02_record_rainfall.png')
piv = cl.pivot(index='month', columns='city', values='record_rain_mm'); print('Month with the heaviest rainfall record:', piv.idxmax().astype(str).to_dict(), '| values (mm):', piv.max().round(0).to_dict())

Month with the heaviest rainfall record: {'Gilgit': 'April', 'Peshawar': 'August'} | values (mm): {'Gilgit': 127.0, 'Peshawar': 451.0}


![02_record_rainfall.png](../figures/02_record_rainfall.png)

### When were the record highs set?
If warming were pushing records upward, recent decades should hold more of them. **Caveat:** each city has only 12 records, and the two records start in different years, so this is suggestive, not proof.

In [10]:
fig, ax = plt.subplots(figsize=(10.5, 4.8)); ax.axvspan(2000, 2025, color=RED, alpha=.07, lw=0)
for city, col, dy in [('Peshawar', BLUE, -.13), ('Gilgit', ORANGE, .13)]:
    d = cl[cl.city==city]; ax.scatter(d.max_year, d.month.cat.codes + dy, s=80, color=col, edgecolor=SURF, linewidth=1.5, label=city, zorder=3)
ax.set_yticks(range(12)); ax.set_yticklabels([m[:3] for m in months]); ax.invert_yaxis(); ax.set_xlim(1875, 2026); ax.set_xlabel('Year the record high was set')
ax.text(2001, 11.55, '2000 onward', color=RED, fontsize=9.5, va='bottom'); ax.legend(loc='upper left', ncol=2); ax.grid(axis='y', visible=False)
ax.set_title('Year each month\'s record high temperature was set'); save('03_record_highs_timeline.png')
rec = cl.assign(recent=cl.max_year>=2000).groupby('city').recent.agg(['sum','size']); print('Record highs set in 2000 or later:', {c: f"{int(r['sum'])} of {int(r['size'])}" for c, r in rec.iterrows()})

Record highs set in 2000 or later: {'Gilgit': '6 of 12', 'Peshawar': '3 of 12'}


![03_record_highs_timeline.png](../figures/03_record_highs_timeline.png)

## 3. Pakistan's temperature since 1870
`temp.csv` has yearly average temperatures for stations in many countries (cleaned in section 1). Pakistan has up to 6 distinct station readings per year (after removing exact repeats), averaged into one national value.

In [11]:
print('Pakistan:', pk_y.year.min(), '-', pk_y.year.max(), '|', len(pk_y), 'years, up to', pk_y.n_obs.max(), 'distinct readings per year')

Pakistan: 1796 - 2013 | 212 years, up to 6 distinct readings per year


In [12]:
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.plot(pk_y.year, pk_y.temp, color=BLUE, lw=1, alpha=.55); ax.plot(pk_y.year, pk_y.temp.rolling(10, center=True).mean(), color=BLUE, lw=2.6)
ax.axvspan(pk_y.year.min(), 1870, color=GREY, alpha=.3, lw=0); ax.text(1800, 19.0, 'Before 1870: implausible\nswings, excluded', color=MUTED, fontsize=10)
ax.set_ylabel('Annual mean temperature (°C)'); ax.set_title('Pakistan\'s average temperature, full record: early years are unreliable'); save('04_pakistan_temperature_full_record.png')

![04_pakistan_temperature_full_record.png](../figures/04_pakistan_temperature_full_record.png)

**Data-quality flag.** Before about 1870 the national mean swings between roughly 17.5 °C and 27.5 °C, including a drop of about 6 °C in the 1850s-60s. That is not a plausible climate signal for a whole country, and most likely reflects sparse or inconsistent early records. All trend tests therefore use **1870 onward**.

In [13]:
d = pk_y[pk_y.year>=1870].reset_index(drop=True); base = d[(d.year>=1900)&(d.year<=1929)].temp.mean(); d['anom'] = d.temp - base
print(f'Reference period 1900-1929 mean: {base:.2f} °C')
res = mk.original_test(d[d.year>=1900].temp.values); s1900 = res.slope*10
def pettitt(v):
    # Pettitt (1979) change-point test: returns (number of points before the shift, approximate p-value)
    v = np.asarray(v); n = len(v); U = np.array([np.sign(v[:k+1, None] - v[None, k+1:]).sum() for k in range(n-1)])
    K = np.abs(U).max(); return int(np.abs(U).argmax()) + 1, min(1.0, 2*np.exp(-6*K**2/(n**3 + n**2)))
cp_idx, cp_p = pettitt(d.temp.values); cp_year = int(d.year.iloc[cp_idx])
print(f'Mann-Kendall 1900-2013: {res.trend}, p = {res.p:.2g}, Sen slope = {res.slope:.4f} °C/yr = {s1900:.2f} °C per decade')
print(f'Pettitt change-point test (1870-2013): most likely shift around {cp_year}, p = {cp_p:.2g}')
print(f'Mean before / after {cp_year}: {d[d.year<cp_year].temp.mean():.2f} / {d[d.year>=cp_year].temp.mean():.2f} °C')

Reference period 1900-1929 mean: 23.64 °C
Mann-Kendall 1900-2013: increasing, p = 6.1e-14, Sen slope = 0.0108 °C/yr = 0.11 °C per decade
Pettitt change-point test (1870-2013): most likely shift around 1940, p = 2.2e-11
Mean before / after 1940: 23.61 / 24.23 °C


In [14]:
fig = plt.figure(figsize=(12, 6.4)); gs = GridSpec(2, 1, height_ratios=[1, 3.3], hspace=.06)
ax0 = fig.add_subplot(gs[0]); lim = np.abs(d.anom).max()
ax0.imshow(d.anom.values[None, :], aspect='auto', cmap=DIV, vmin=-lim, vmax=lim, extent=[d.year.min(), d.year.max()+1, 0, 1]); ax0.axis('off')
ax0.set_title('Pakistan\'s warming stripes: each bar is one year, blue = cooler and red = warmer than 1900-1929', loc='left', fontsize=12.5, fontweight='bold')
ax1 = fig.add_subplot(gs[1], sharex=ax0); ax1.set_xlim(d.year.min(), d.year.max()+1)
ax1.scatter(d.year, d.temp, s=16, color=BLUE, alpha=.5, edgecolor='none'); ax1.plot(d.year, d.temp.rolling(10, center=True).mean(), color=BLUE, lw=2.6, label='10-year average')
s = d[d.year>=1900]; ax1.plot(s.year, res.intercept + res.slope*(s.year - 1900), color=ORANGE, lw=2.4, ls='--', label=f'Trend since 1900: +{s1900:.2f} °C per decade')
ax1.axvline(cp_year, color=MUTED, lw=1, ls=':'); ax1.text(cp_year-1.5, d.temp.min()+.05, f'Likely shift ~{cp_year}', ha='right', color=MUTED, fontsize=9.5)
ax1.set_ylabel('Annual mean temperature (°C)'); ax1.legend(loc='upper left'); ax1.grid(axis='x', visible=False)
plt.savefig(FIG+'00_hero.png', dpi=170); plt.savefig(FIG+'05_warming_stripes_trend.png', dpi=160); plt.show()

![05_warming_stripes_trend.png](../figures/05_warming_stripes_trend.png)

### Trend strength depends on the period

In [15]:
rows = []
for a in [1870, 1900, 1950, 1980]:
    r_ = mk.original_test(d[d.year>=a].temp.values); rows.append(dict(period=f'{a}-2013', years=len(d[d.year>=a]), trend=r_.trend, p_value=float(f'{max(r_.p,1e-16):.2g}'), sen_slope_C_per_decade=round(r_.slope*10, 2)))
mk_t = pd.DataFrame(rows); display(mk_t)
fig, ax = plt.subplots(figsize=(8, 3.6)); ax.bar(mk_t.period, mk_t.sen_slope_C_per_decade, color=BLUE, width=.55)
for i, v in enumerate(mk_t.sen_slope_C_per_decade): ax.text(i, v+.008, f'+{v:.2f}', ha='center', color=INK, fontsize=10.5)
ax.set_ylabel('Warming rate (°C per decade)'); ax.set_title('The recent warming rate is several times the long-run rate'); ax.grid(axis='x', visible=False); save('06_warming_rate_by_period.png')
dec = d.assign(decade=(d.year//10)*10).query('decade>=1900').groupby('decade').temp.mean(); print(f'Decade mean: {dec.iloc[0]:.2f} °C in the 1900s to {dec.iloc[-1]:.2f} °C in the 2010s (+{dec.iloc[-1]-dec.iloc[0]:.2f} °C)')

,period,years,trend,p_value,sen_slope_C_per_decade
0,1870-2013,144,increasing,1.000000e-16,0.09
1,1900-2013,114,increasing,6.100000e-14,0.11
2,1950-2013,64,increasing,3.700000e-08,0.20
3,1980-2013,34,increasing,4.900000e-04,0.30


Decade mean: 23.64 °C in the 1900s to 24.96 °C in the 2010s (+1.32 °C)


![06_warming_rate_by_period.png](../figures/06_warming_rate_by_period.png)

### Robustness: autocorrelation and confidence intervals
Yearly temperatures are correlated with their neighbours, which can make the plain Mann-Kendall test overconfident. We re-test with the Hamed-Rao autocorrelation correction and report 95% confidence intervals for the Theil-Sen slope.

In [16]:
from scipy.stats import theilslopes
rows = []
for a in [1870, 1900, 1950, 1980]:
    s_ = d[d.year>=a]; plain = mk.original_test(s_.temp.values); hr = mk.hamed_rao_modification_test(s_.temp.values); sl, ic, lo, up = theilslopes(s_.temp.values, s_.year.values)
    rows.append(dict(period=f'{a}-2013', plain_p=float(f'{max(plain.p,1e-16):.2g}'), adjusted_p=float(f'{hr.p:.2g}'), still_significant=bool(hr.p<.05), slope_C_per_decade=round(sl*10,2), ci95_low=round(lo*10,2), ci95_high=round(up*10,2)))
rb = pd.DataFrame(rows); display(rb)
fig, ax = plt.subplots(figsize=(8, 3.6)); yy = np.arange(len(rb))[::-1]
ax.errorbar(rb.slope_C_per_decade, yy, xerr=[rb.slope_C_per_decade-rb.ci95_low, rb.ci95_high-rb.slope_C_per_decade], fmt='o', color=BLUE, ecolor=BLUE, elinewidth=2.5, capsize=4, ms=8)
ax.set_yticks(yy); ax.set_yticklabels(rb.period); ax.axvline(0, color=MUTED, lw=.8); ax.set_xlabel('Warming rate, °C per decade (95% confidence interval)'); ax.grid(axis='y', visible=False)
ax.set_title('Every interval excludes zero, even after the correction' if (rb.ci95_low>0).all() else 'Not every interval excludes zero'); save('06b_warming_rate_confidence.png')
print('Lag-1 autocorrelation of annual temperature since 1870:', round(d.temp.autocorr(1), 2))

,period,plain_p,adjusted_p,still_significant,slope_C_per_decade,ci95_low,ci95_high
0,1870-2013,1.000000e-16,6.700000e-09,True,0.09,0.07,0.10
1,1900-2013,6.100000e-14,8.500000e-07,True,0.11,0.08,0.13
2,1950-2013,3.700000e-08,3.700000e-08,True,0.20,0.15,0.26
3,1980-2013,4.900000e-04,4.900000e-04,True,0.30,0.15,0.44


Lag-1 autocorrelation of annual temperature since 1870: 0.57


![06b_warming_rate_confidence.png](../figures/06b_warming_rate_confidence.png)

**Reading it:** the trend is significant in every period, and the rate grows when you look at recent decades. This says the temperature record is rising, not what caused it. The data is also not corrected for changes in stations or instruments.

## 4. Crop yields: distributions and Pakistan's crops
`yield_df.csv` combines FAO yield and pesticide data (1990-2013) with temperature for 101 countries. Its quality problems and fixes are listed in section 1.

In [17]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.3))
order = yc.groupby('Item').yield_t_ha.median().sort_values().index
sns.boxplot(data=yc, y='Item', x='yield_t_ha', order=order, ax=ax[0], showfliers=False, color='#9ec5f4', linecolor=BLUE, linewidth=1.2); ax[0].set_xlabel('Yield (t/ha), all countries'); ax[0].set_ylabel(''); ax[0].set_title('Yields differ hugely by crop')
sns.histplot(np.log10(yc.yield_hgha), bins=40, ax=ax[1], color=BLUE, edgecolor=SURF); ax[1].set_xlabel('log10 of yield (hg/ha)'); ax[1].set_title('Log scale tames the skew')
plt.tight_layout(); save('07_yield_distribution.png'); print(f'Skewness: raw {yc.yield_hgha.skew():.2f} to {np.log10(yc.yield_hgha).skew():.2f} after log10')

Skewness: raw 2.08 to -0.22 after log10


![07_yield_distribution.png](../figures/07_yield_distribution.png)

### Pakistan's crops, 1990-2013

In [18]:
p = yc[yc.Area=='Pakistan'].sort_values(['Item','Year']).copy()
from scipy.stats import theilslopes
res = []
for it, g in p.groupby('Item'):
    r_ = mk.original_test(g.yield_t_ha.values); hr_ = mk.hamed_rao_modification_test(g.yield_t_ha.values); sl, ic, lo, up = theilslopes(g.yield_t_ha.values, g.Year.values)
    res.append(dict(crop=it, mean_yield_t_ha=round(g.yield_t_ha.mean(),2), trend=r_.trend, p_value=float(f'{r_.p:.2g}'), adjusted_p=float(f'{hr_.p:.2g}'), slope_t_ha_per_yr=round(sl,3), ci95_low=round(lo,3), ci95_high=round(up,3), pct_per_yr=round(100*sl/g.yield_t_ha.mean(),1), intercept=ic, slope_raw=sl))
pk_mk = pd.DataFrame(res).sort_values('pct_per_yr', ascending=False).reset_index(drop=True); display(pk_mk.drop(columns=['intercept','slope_raw']))

,crop,mean_yield_t_ha,trend,p_value,adjusted_p,slope_t_ha_per_yr,ci95_low,ci95_high,pct_per_yr
0,Maize,2.44,increasing,1.700000e-09,7.100000e-06,0.134,0.111,0.154,5.5
1,Potatoes,15.93,increasing,6.900000e-07,6.900000e-07,0.500,0.412,0.561,3.1
2,"Rice, paddy",3.01,increasing,2.200000e-08,2.200000e-08,0.056,0.046,0.067,1.9
3,Wheat,2.32,increasing,3.900000e-08,3.900000e-08,0.042,0.036,0.050,1.8
4,Sweet potatoes,10.48,no trend,1.500000e-01,3.000000e-01,0.058,-0.014,0.095,0.6
5,Sorghum,0.61,increasing,2.200000e-04,2.200000e-04,0.002,0.001,0.002,0.3
6,Soybeans,0.93,no trend,9.600000e-01,9.700000e-01,-0.001,-0.033,0.016,-0.1


In [19]:
key = ['Wheat','Rice, paddy','Maize','Potatoes']; fig, axes = plt.subplots(1, 4, figsize=(14, 3.7))
for ax, it in zip(axes, key):
    g = p[p.Item==it]; m = pk_mk.set_index('crop').loc[it]; ax.plot(g.Year, g.yield_t_ha, color=BLUE, lw=2.2, marker='o', ms=3.5)
    ax.plot(g.Year, m.intercept + m.slope_raw*g.Year, color=ORANGE, lw=2, ls='--')
    ax.set_title(it.replace(', paddy','')); ax.set_xticks([1990, 2000, 2010]); ax.text(.04, .92, f'{m.pct_per_yr:+.1f}% per year', transform=ax.transAxes, color=INK, fontsize=10, va='top', fontweight='bold')
axes[0].set_ylabel('Yield (t/ha)'); fig.suptitle('Pakistan\'s main crops: yield (blue) and Sen trend (orange dashed)', x=.01, ha='left', fontweight='bold', fontsize=13.5, y=1.04)
plt.tight_layout(); save('08_pakistan_crop_trends.png')
sig = pk_mk[pk_mk.adjusted_p<.05].crop.tolist(); non = pk_mk[pk_mk.adjusted_p>=.05].crop.tolist(); print('Significant increase (autocorrelation-adjusted):', sig, '| no significant trend:', non)

Significant increase (autocorrelation-adjusted): ['Maize', 'Potatoes', 'Rice, paddy', 'Wheat', 'Sorghum'] | no significant trend: ['Sweet potatoes', 'Soybeans']


![08_pakistan_crop_trends.png](../figures/08_pakistan_crop_trends.png)

In [20]:
piv = p.pivot(index='Item', columns='Year', values='yield_t_ha'); idx = piv.div(piv.mean(axis=1), axis=0).loc[pk_mk.crop]
fig, ax = plt.subplots(figsize=(12, 3.6)); im = ax.imshow(idx.values, aspect='auto', cmap=DIV, norm=TwoSlopeNorm(vmin=.3, vcenter=1, vmax=1.7), extent=[1989.5, 2013.5, len(idx), 0])
ax.set_yticks(np.arange(len(idx))+.5); ax.set_yticklabels(idx.index); ax.grid(False); ax.spines[:].set_visible(False)
cb = fig.colorbar(im, ax=ax, pad=.01); cb.set_label('Yield relative to crop average (1.0 = average)'); ax.set_title('Which crops improved? Blue = below the crop\'s own average, red = above (sorted by growth rate)'); save('09_pakistan_yield_heatmap.png')

![09_pakistan_yield_heatmap.png](../figures/09_pakistan_yield_heatmap.png)

**A caution on maize:** its yield was flat near 1.5 t/ha until about 2002, then jumped to above 4 t/ha by 2013. A straight trend line therefore fits it poorly, and the +5.8% per year figure averages over a flat period and a steep one. The data cannot tell us why (new varieties, area changes or reporting changes are all possible).

### How does Pakistan compare?
Pakistan against other countries on **wheat**, its staple. India is its neighbour with a similar growing environment, and Egypt is another irrigated, arid-climate producer.

In [21]:
wh = yc[yc.Item=='Wheat'].groupby('Area').yield_t_ha.mean().sort_values(ascending=False); n = len(wh); rank = int(wh.index.get_loc('Pakistan'))+1
show = pd.concat([wh.head(10), wh[['India','Egypt','Pakistan']]]).loc[lambda s: ~s.index.duplicated()].sort_values()
fig, ax = plt.subplots(figsize=(8.5, 5)); cols = [ORANGE if a=='Pakistan' else (BLUE if a in ('India','Egypt') else GREY) for a in show.index]
ax.barh(show.index, show.values, color=cols, height=.7); ax.grid(axis='y', visible=False)
for i, v in enumerate(show.values): ax.text(v+.08, i, f'{v:.1f}', va='center', fontsize=9.5, color=INK)
ax.set_xlabel('Mean wheat yield 1990-2013 (t/ha)'); ax.set_title(f'Wheat yield: Pakistan ranks {rank} of {n} countries'); save('10_wheat_ranking.png')
gap = wh.head(10).mean()/wh['Pakistan']; print(f'Top-10 average wheat yield is {gap:.1f}x Pakistan\'s ({wh.head(10).mean():.1f} vs {wh["Pakistan"]:.1f} t/ha)')

Top-10 average wheat yield is 3.1x Pakistan's (7.3 vs 2.3 t/ha)


![10_wheat_ranking.png](../figures/10_wheat_ranking.png)

In [22]:
cmp = [c for c in ['Pakistan','India','Egypt'] if c in yc[yc.Item=='Wheat'].Area.unique()]; cols = {'Pakistan':ORANGE,'India':BLUE,'Egypt':AQUA}
fig, ax = plt.subplots(figsize=(9.5, 4.3))
for c in cmp:
    g = yc[(yc.Area==c)&(yc.Item=='Wheat')].sort_values('Year'); idx_ = 100*g.yield_t_ha/g.yield_t_ha.iloc[0]; ax.plot(g.Year, idx_, color=cols[c], lw=2.4)
    ax.text(g.Year.max()+.4, idx_.iloc[-1], f'{c} {idx_.iloc[-1]:.0f}', color=INK, va='center', fontsize=10)
ax.axhline(100, color=MUTED, lw=.8); ax.set_xlim(1990, 2016); ax.set_ylabel('Wheat yield index (1990 = 100)'); ax.set_title('Pakistan\'s wheat yield grew about as fast as India\'s, from a low base'); ax.grid(axis='x', visible=False); save('11_wheat_growth_index.png')

![11_wheat_growth_index.png](../figures/11_wheat_growth_index.png)

### Does temperature explain yield changes?
Both yields and temperature rise over time, so a raw correlation looks strong **even if the two are unrelated**. A fairer check correlates *year-to-year changes*.

In [23]:
raw = p.groupby('Item').apply(lambda g: g.yield_t_ha.corr(g.avg_temp, method='spearman'))
dif = p.groupby('Item').apply(lambda g: g.yield_t_ha.diff().corr(g.avg_temp.diff(), method='spearman'))
cc = pd.DataFrame({'raw':raw, 'change':dif}).sort_values('raw'); yy = np.arange(len(cc)); fig, ax = plt.subplots(figsize=(8.5, 4.6)); h = .38
ax.barh(yy+h/2, cc.raw, h-.04, color=GREY, label='Raw levels (misleading: shared time trend)'); ax.barh(yy-h/2, cc['change'], h-.04, color=BLUE, label='Year-to-year changes (fairer)')
ax.set_yticks(yy); ax.set_yticklabels(cc.index); ax.axvline(0, color=MUTED, lw=.8); ax.set_xlabel('Spearman correlation of yield with temperature'); ax.legend(loc='lower right'); ax.grid(axis='y', visible=False)
ax.set_title('Yield and temperature: the apparent link mostly disappears'); save('12_yield_temperature_correlation.png')
print('Raw range:', f'{cc.raw.min():.2f} to {cc.raw.max():.2f}', '| year-to-year change range:', f'{cc["change"].min():.2f} to {cc["change"].max():.2f}', '| n = 23 years per crop')

Raw range: 0.11 to 0.67 | year-to-year change range: -0.28 to 0.53 | n = 23 years per crop


![12_yield_temperature_correlation.png](../figures/12_yield_temperature_correlation.png)

## 5. Conclusions and limitations
**What the data supports**
- Pakistan's average temperature rose significantly since 1870, and the rate is higher in recent decades. A change-point test places the largest single step in the mean at about 1940 (23.6 to 24.2 °C), yet the series keeps warming, and faster after 1980, so one step does not describe it fully.
- Gilgit's record highs are more recent than Peshawar's, which fits warming but rests on only 12 values per city.
- Most of Pakistan's crop yields rose from 1990 to 2013, led by maize and potatoes. Pakistan's wheat yield grew about as fast as India's (both about +50% since 1990) but remains well below the top producers.
- There is no clear evidence in this data that temperature drives yield changes.

**Limits to keep in mind**
- Peshawar and Gilgit are record tables, not yearly series, so no trend tests were possible for them.
- The temperature series is not corrected for station or instrument changes, and the pre-1870 record is unreliable.
- Rainfall in the crop file is constant per country and was not used.
- Crop data has only 23-24 years, which limits statistical power. Correlation and trend tests do not establish causes.

**Next steps:** add real yearly rainfall for Pakistan, add basin-level hydrology data, and test detrended or lagged yield-climate models.